In [ ]:
from pyspark.sql.functions import * 

orders_df = spark.table("olist_.bronze_orders")

silver_orders = (
    orders_df
    .withColumn("order_purchase_timestamp",
                to_timestamp("order_purchase_timestamp"))
    .withColumn("order_approved_at",
                to_timestamp("order_approved_at"))
    .withColumn("order_delivered_carrier_date",
                to_timestamp("order_delivered_carrier_date"))
    .withColumn("order_delivered_customer_date",
                to_timestamp("order_delivered_customer_date"))
    .withColumn("order_estimated_delivery_date",
                to_timestamp("order_estimated_delivery_date"))
)

silver_orders.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("olist_.silver_orders")


In [ ]:
df_customers = spark.table("olist_.bronze_customers")

silver_customers = (
    df_customers
    .withColumn("customer_city", initcap(lower(trim(col("customer_city")))))
    .withColumn("customer_state", upper(trim(col("customer_state"))))
)

silver_customers.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("olist_.silver_customers")

In [ ]:
df_payments = spark.table("olist_.bronze_payments")

silver_payments = (
    df_payments
    .filter(col("payment_value") >= 0)
)

silver_payments.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("olist_.silver_payments")

In [ ]:
items_df = spark.table("olist_.bronze_order_items")

silver_items = (
    items_df
    .withColumn("shipping_limit_date", to_timestamp(col("shipping_limit_date")))
    .dropDuplicates(["order_id", "order_item_id"])
)

silver_items.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("olist_.silver_order_items")

In [ ]:
reviews_df = spark.table("olist_.bronze_reviews")

silver_reviews = (
    reviews_df
    .withColumn("review_creation_date", to_timestamp(col("review_creation_date")))
    .withColumn("review_answer_timestamp", to_timestamp(col("review_answer_timestamp")))
    .filter(col("review_score").between(1, 5))
)

silver_reviews.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("olist_.silver_reviews")

In [ ]:
products = spark.table("olist_.bronze_products")
silver_products = (
    products
    .dropDuplicates(["product_id"])
    .withColumn("product_category_name", lower(trim(col("product_category_name"))))
    .filter(col("product_weight_g").isNull() | (col("product_weight_g") >= 0))
    .filter(col("product_length_cm").isNull() | (col("product_length_cm") >= 0))
    .filter(col("product_height_cm").isNull() | (col("product_height_cm") >= 0))
    .filter(col("product_width_cm").isNull() | (col("product_width_cm") >= 0))
)
silver_products.write.format("delta").mode("overwrite").saveAsTable("olist_.silver_products")

In [ ]:
sellers = spark.table("olist_.bronze_sellers")
silver_sellers = (
    sellers
    .dropDuplicates(["seller_id"])
    .withColumn("seller_city", lower(trim(col("seller_city"))))
    .withColumn("seller_state", upper(trim(col("seller_state"))))
)
silver_sellers.write.format("delta").mode("overwrite").saveAsTable("olist_.silver_sellers")

In [ ]:
geo = spark.table("olist_.bronze_geolocation")
silver_geo = (
    geo
    .dropDuplicates(["geolocation_zip_code_prefix", "geolocation_lat", "geolocation_lng"])
    .withColumn("geolocation_city", lower(trim(col("geolocation_city"))))
    .withColumn("geolocation_state", upper(trim(col("geolocation_state"))))
)
silver_geo.write.format("delta").mode("overwrite").saveAsTable("olist_.silver_geolocation")

In [ ]:
cat = spark.table("olist_.bronze_category_translation")
silver_cat = (
    cat
    .dropDuplicates(["product_category_name"])
    .withColumn("product_category_name", lower(trim(col("product_category_name"))))
    .withColumn("product_category_name_english", lower(trim(col("product_category_name_english"))))
)
silver_cat.write.format("delta").mode("overwrite").saveAsTable("olist_.silver_category_translation")